# Автоматическая генерация признаков для ранжирования рекомендаций (KION)

**Вопрос исследования.** Ранкер на градиентном бустинге обычно работает на признаках, которые придумали инженеры: окна, агрегаты, связи пользователя с атрибутами объекта. Можно ли заменить ручной перебор автоматическим поиском по пространству признаков и не потерять в качестве?

**Что сделано в ноутбуке:**

1. Данные KION и их особенности, которые влияют на постановку.
2. Постановка: предсказать, какие новые тайтлы пользователь посмотрит на следующей неделе. Разбиение по времени, генерация кандидатов, метрики.
3. Бейзлайн: LightGBM на ручных признаках.
4. Автоматическая генерация: грамматика признаков, которая компилируется в SQL, и две стратегии поиска по ней — сетка с одномерным фильтром и жадный отбор.
5. LLM-агент, который предлагает кандидатов для жадного отбора вместо случайного генератора, с тем же бюджетом.
6. Сравнение на тестовой неделе, LogLoss, проверка на второй тестовой неделе и на пяти seed, вычислительные затраты.
7. Выводы и ограничения.

Все подходы используют одно разбиение, один набор кандидатов и одну историю событий. Признаки и гиперпараметры подбираются на валидационной неделе, тест используется один раз в конце.

## 0. Окружение

Тяжёлые шаги кэшируются в `cache/` в parquet: при повторном запуске ноутбук читает готовые таблицы. Чтобы пересчитать шаг, удалите соответствующий файл.

Запуск на Kaggle. Следующая ячейка локально ничего не делает. На Kaggle она:

1. ставит версии библиотек, которых нет в образе;
2. клонирует репозиторий проекта: в нём лежат результаты поисков и ответы LLM, поэтому ноутбук не повторяет полтора часа поиска и не требует ключа API;
3. находит данные: подключённый датасет с `interactions.csv` или, если такого нет, скачивает полный архив KION с GitHub (нужен включённый интернет в настройках ноутбука).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'polars==2.0.0', 'duckdb==2.0.0',
                    'lightgbm==4.7.0', 'openai', 'python-dotenv'], check=True)
    repo = Path('/kaggle/working/recsys-auto-features')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/404iq/recsys-auto-features.git', str(repo)], check=True)
    os.chdir(repo)                       # дальше пути относительные, как при локальном запуске
    found = list(Path('/kaggle/input').rglob('interactions.csv'))
    if found:
        os.environ['KION_DATA_DIR'] = str(found[0].parent)
    else:
        url = 'https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip'
        subprocess.run(['wget', '-q', url, '-O', 'kion.zip'], check=True)
        subprocess.run(['unzip', '-q', '-j', '-o', 'kion.zip', 'data_original/*.csv', '-d', 'data'], check=True)

In [ ]:
import os
os.environ.setdefault('MIMALLOC_PURGE_DELAY', '0')   # аллокатор polars сразу возвращает память системе: на ноутбуке её мало

import gc
import json
import time
from contextlib import contextmanager
from datetime import date, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import scipy.sparse as sp

DATA_DIR = Path(os.environ.get('KION_DATA_DIR', 'data'))
CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)

SEED = 42
N_USERS = 100_000          # размер сэмпла пользователей
TEST_START = date(2021, 8, 16)
K = 10                     # глубина метрик ранжирования

COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']
plt.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': '#52514e',
    'xtick.color': '#52514e', 'ytick.color': '#52514e',
    'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.prop_cycle': plt.cycler(color=COLORS), 'figure.dpi': 110,
})
pl.Config.set_tbl_rows(12)

TIMINGS = {}
TIMINGS_PATH = CACHE_DIR / 'timings.json'
FIRST_RUN = json.loads(TIMINGS_PATH.read_text(encoding='utf-8')) if TIMINGS_PATH.exists() else {}


@contextmanager
def timer(name):
    """Замер времени. Время первого прогона (без кэша) сохраняется для таблицы затрат."""
    t0 = time.perf_counter()
    yield
    TIMINGS[name] = time.perf_counter() - t0
    if name not in FIRST_RUN:
        FIRST_RUN[name] = TIMINGS[name]
        TIMINGS_PATH.write_text(json.dumps(FIRST_RUN, ensure_ascii=False, indent=1), encoding='utf-8')
    print(f'{name}: {TIMINGS[name]:.1f} с')


def cached(name, build):
    """Читает cache/<name>.parquet или строит таблицу и сохраняет её."""
    path = CACHE_DIR / f'{name}.parquet'
    if path.exists():
        return pl.read_parquet(path)
    df = build()
    df.write_parquet(path)
    return df

## 1. Данные

Датасет KION от МТС: просмотры онлайн-кинотеатра с 13 марта по 22 августа 2021 года.

- `interactions.csv`: `user_id, item_id, last_watch_dt, total_dur, watched_pct`;
- `users.csv`: возраст, доход, пол, наличие детей (категории);
- `items.csv`: тип контента, год, жанры, страны, режиссёры, актёры, возрастной рейтинг и т. д.

В репозитории [KION_DATASET](https://github.com/irsafilo/KION_DATASET) архив `interactions.csv.zip` обрезан: в нём ровно 50 МиБ и около 1.6 млн строк. Полный файл (5 476 251 строка) лежит внутри `data_original.zip`, его и используем. Проверка ниже ловит неполную версию.

In [ ]:
interactions = (
    pl.read_csv(DATA_DIR / 'interactions.csv', try_parse_dates=True)
    .rename({'last_watch_dt': 'dt'})
)
users = pl.read_csv(DATA_DIR / 'users.csv')
items = pl.read_csv(DATA_DIR / 'items.csv')

assert interactions.height == 5_476_251, 'interactions.csv неполный: возьмите его из data_original.zip'
print(f'взаимодействий: {interactions.height:,}')
print(f'пользователей в логах: {interactions["user_id"].n_unique():,}, с анкетой: {users.height:,}')
print(f'объектов в логах: {interactions["item_id"].n_unique():,}, в каталоге: {items.height:,}')
print(f'период: {interactions["dt"].min()} — {interactions["dt"].max()}')

### 1.1. Одна строка на пару пользователь–объект

Главная особенность этих логов определяет многое в постановке.

In [ ]:
n_pairs = interactions.select('user_id', 'item_id').unique().height
print(f'строк: {interactions.height:,}, уникальных пар: {n_pairs:,}')
print(f'пропусков в watched_pct: {interactions["watched_pct"].null_count()}')

Каждая пара встречается ровно один раз. `last_watch_dt` — дата **последнего** просмотра, `total_dur` и `watched_pct` накоплены за всё время. Что из этого следует:

1. **Повторное потребление не видно.** Признаки прямой связи («сколько раз пользователь смотрел этот тайтл») посчитать нельзя. Персональный сигнал придётся брать из косвенных связей: жанры, режиссёры, актёры, страны.
2. **Отсечка по дате не даёт утечки, но теряет часть истории.** Если `last_watch_dt < T`, после момента `T` пользователь этот тайтл не смотрел, значит `total_dur` и `watched_pct` целиком относятся к прошлому. Если `last_watch_dt ≥ T`, строку из истории приходится выкинуть целиком, хотя часть просмотра могла быть раньше `T`. Ограничение есть в данных, утечки нет.
3. **Таргет не отличает первый просмотр от досмотра.** Тайтл, начатый до `T` и досмотренный после, попадёт в таргет как «новый». Отделить такие случаи по этим данным нельзя.

Показов в логах тоже нет: есть только факт просмотра и его глубина. Типы событий придётся выводить из `watched_pct` (бросил, досмотрел).

828 пропусков в `watched_pct` заполняем нулём.

In [ ]:
interactions = interactions.with_columns(pl.col('watched_pct').fill_null(0.0))

weekly = (
    interactions.with_columns(week=pl.col('dt').dt.truncate('1w'))
    .group_by('week').len().sort('week')
)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].bar(weekly['week'].to_list(), weekly['len'].to_numpy() / 1e3, width=5, color=COLORS[0])
axes[0].set_title('Просмотров в неделю, тыс.', loc='left')
counts, edges = np.histogram(interactions['watched_pct'].to_numpy(), bins=50)
axes[1].bar(edges[:-1], counts / 1e6, width=np.diff(edges) * 0.9, align='edge', color=COLORS[0])
axes[1].set_title('Распределение watched_pct, млн просмотров', loc='left')
axes[1].set_xlabel('% просмотренного')
fig.tight_layout()
plt.show()

Объём просмотров растёт к концу периода: в последнюю неделю примерно в четыре раза больше просмотров, чем в марте–апреле. Часть роста может быть артефактом формата: при повторном просмотре строка «переезжает» на более позднюю дату. Первая и последняя «недели» на графике неполные по календарю, но последняя (16–22 августа) содержит все 7 дней, поэтому её берём в тест.

У `watched_pct` два пика, около 0 и ровно 100: большая часть просмотров либо брошена в начале, либо досмотрена. Это естественные границы для типов событий.

In [ ]:
item_pop = interactions.group_by('item_id').len().sort('len', descending=True)
top1 = item_pop.head(int(0.01 * item_pop.height))['len'].sum() / interactions.height
top10 = item_pop.head(int(0.10 * item_pop.height))['len'].sum() / interactions.height
print(f'доля просмотров у топ-1% объектов: {top1:.1%}, у топ-10%: {top10:.1%}')

user_act = interactions.group_by('user_id').len()['len']
print(f'просмотров на пользователя: медиана {user_act.median():.0f}, среднее {user_act.mean():.1f}, 99-й перцентиль {user_act.quantile(0.99):.0f}')

null_share = items.select(
    [(pl.col(c).null_count() / items.height).alias(c)
     for c in ['genres', 'countries', 'directors', 'actors', 'studios', 'keywords']]
)
null_share

Популярность сильно сконцентрирована: на 1% тайтлов приходится почти половина просмотров, на 10% — 85%. У половины пользователей всего 1–2 просмотра за пять месяцев. Значит, популярность будет сильным бейзлайном, и персонализация должна дать прирост сверх неё.

У `studios` 93% пропусков, этот атрибут для связей не используем. Жанры, страны, режиссёры и актёры заполнены хорошо.

## 2. Постановка задачи

**Задача:** по истории до даты `T` выбрать для пользователя 10 тайтлов, которые он посмотрит в неделю `[T, T+7)`. Из-за одной строки на пару в таргет попадают только тайтлы, которых нет в истории пользователя до `T`, поэтому из кандидатов просмотренное тоже исключаем.

Схема двухстадийная, как в продовых системах:

1. Генераторы кандидатов (популярность, ItemKNN) дают по ~170 тайтлов на пользователя.
2. Ранкер (LightGBM) упорядочивает кандидатов по признакам, посчитанным по истории до `T`.

### 2.1. Сэмпл пользователей

Берём случайные 100 тыс. пользователей и **всю** их историю. Все статистики, включая популярность и item-item сходство, считаем только по сэмплу: так эксперимент честно имитирует датасет меньшего размера. На 7.7 ГБ RAM это даёт запас для таблиц кандидатов с сотнями признаков.

In [ ]:
rng = np.random.default_rng(SEED)
all_users = interactions['user_id'].unique().sort().to_numpy()
sample_users = rng.choice(all_users, size=N_USERS, replace=False)

log = interactions.filter(pl.col('user_id').is_in(sample_users))

compare = pl.DataFrame({
    'метрика': ['просмотров на пользователя', 'доля досмотров (≥80%)', 'объектов'],
    'весь датасет': [
        interactions.height / len(all_users),
        (interactions['watched_pct'] >= 80).mean(),
        interactions['item_id'].n_unique(),
    ],
    'сэмпл': [
        log.height / N_USERS,
        (log['watched_pct'] >= 80).mean(),
        log['item_id'].n_unique(),
    ],
})
print(f'строк в сэмпле: {log.height:,}')
compare

### 2.2. Разбиение по времени

Используем скользящие недели. Для каждой целевой недели признаки и кандидаты строятся только по событиям до её начала.

| Фолд | История | Неделя таргета | Назначение |
|---|---|---|---|
| `train` | до 02.08 | 02.08–08.08 | обучение ранкера |
| `val` | до 09.08 | 09.08–15.08 | подбор признаков, гиперпараметров, early stopping |
| `test` | до 16.08 | 16.08–22.08 | одна финальная оценка |

Финальная модель для теста обучается на неделе `val` (признаки до 09.08, таргет 09.08–15.08) с параметрами, найденными на `train → val`. Так модель видит самые свежие данные, а в таргет теста при этом ничего не просачивается.

Ранжируем только **тёплых** пользователей, у которых до `T` есть хотя бы один просмотр. Холодным признакам пользователя не из чего взяться, им показываем популярное. Эта доля считается отдельно и одинакова для всех ранкеров.

In [ ]:
del interactions          # дальше работаем только с сэмплом, полный лог занимает лишнюю память

FOLDS = {
    'train': TEST_START - timedelta(days=14),
    'val': TEST_START - timedelta(days=7),
    'test': TEST_START,
}


def split(log, start):
    """История строго до start и таргет-неделя [start, start+7) для тёплых пользователей."""
    hist = log.filter(pl.col('dt') < start)
    target = log.filter((pl.col('dt') >= start) & (pl.col('dt') < start + timedelta(days=7)))
    warm = target.filter(pl.col('user_id').is_in(hist['user_id'].unique().implode()))
    return hist, warm, target


rows = []
for name, start in FOLDS.items():
    hist, warm, target = split(log, start)
    rows.append({
        'фолд': name, 'начало недели': str(start),
        'строк истории': hist.height,
        'пользователей в таргете': target['user_id'].n_unique(),
        'из них тёплых': warm['user_id'].n_unique(),
        'позитивов у тёплых': warm.height,
    })
pl.DataFrame(rows)

### 2.3. Генерация кандидатов

Кандидаты одинаковы для всех ранкеров, поэтому их качество задаёт потолок для всех сравнений. Используем два источника:

- **популярность** за последние 7 дней до `T`, топ-100 непросмотренных;
- **ItemKNN**: косинусное сходство тайтлов по совместным просмотрам, топ-100 по сумме сходств с историей пользователя. Вклад прошлых просмотров затухает экспоненциально с полупериодом 30 дней, чтобы свежие интересы весили больше.

Ранги и скоры источников сохраняем: дальше они станут признаками ранкера, одинаковыми для ручного и автоматического подходов.

In [ ]:
N_POP, N_KNN, KNN_NEIGHBORS, KNN_HALF_LIFE = 100, 100, 200, 30


def popular_candidates(hist, warm_users, start, n=N_POP):
    pop = (
        hist.filter(pl.col('dt') >= start - timedelta(days=7))
        .group_by('item_id').len().sort('len', descending=True)
        .head(n + 200)                  # запас на уже просмотренные
        .with_columns(pop_score=pl.col('len').cast(pl.Float32))
        .select('item_id', 'pop_score')
    )
    seen = hist.filter(pl.col('user_id').is_in(warm_users.implode())).select('user_id', 'item_id')
    return (
        warm_users.to_frame().join(pop, how='cross')
        .join(seen, on=['user_id', 'item_id'], how='anti')
        .sort(['user_id', 'pop_score'], descending=[False, True])
        .with_columns(pop_rank=pl.int_range(1, pl.len() + 1).over('user_id').cast(pl.Int32))
        .filter(pl.col('pop_rank') <= n)
    )


def prune_rows(S, k):
    """Оставляет k наибольших значений в каждой строке разреженной матрицы."""
    S = S.tocsr()
    indptr, data = S.indptr, S.data
    keep = np.zeros(len(data), dtype=bool)
    for i in range(S.shape[0]):
        s, e = indptr[i], indptr[i + 1]
        if e - s <= k:
            keep[s:e] = True
        else:
            keep[s + np.argpartition(-data[s:e], k)[:k]] = True
    S.data = np.where(keep, data, 0).astype(np.float32)
    S.eliminate_zeros()
    return S


def knn_candidates(hist, warm_users, start, n=N_KNN, batch=500):
    item_ids = hist['item_id'].unique().sort().to_numpy()
    user_ids = hist['user_id'].unique().sort().to_numpy()
    h = hist.with_columns(
        u=pl.col('user_id').replace_strict(user_ids, np.arange(len(user_ids))),
        i=pl.col('item_id').replace_strict(item_ids, np.arange(len(item_ids))),
        w=(0.5 ** ((pl.lit(start) - pl.col('dt')).dt.total_days() / KNN_HALF_LIFE)).cast(pl.Float32),
    )
    shape = (len(user_ids), len(item_ids))
    X = sp.csr_matrix((np.ones(h.height, dtype=np.float32), (h['u'].to_numpy(), h['i'].to_numpy())), shape=shape)
    norms = np.sqrt(np.asarray(X.sum(axis=0)).ravel())
    norms[norms == 0] = 1
    Xn = X @ sp.diags(1 / norms)
    S = (Xn.T @ Xn).tocsr()
    S.setdiag(0)
    S = prune_rows(S, KNN_NEIGHBORS)

    W = sp.csr_matrix((h['w'].to_numpy(), (h['u'].to_numpy(), h['i'].to_numpy())), shape=shape)
    rows_idx = np.searchsorted(user_ids, warm_users.to_numpy())
    out = []
    for b in range(0, len(rows_idx), batch):
        r = rows_idx[b:b + batch]
        Wb = W[r]
        scores = (Wb @ S).toarray()
        scores[Wb.nonzero()] = -np.inf                     # просмотренное не рекомендуем
        top = np.argpartition(-scores, n, axis=1)[:, :n]
        top_scores = np.take_along_axis(scores, top, axis=1)
        out.append(pl.DataFrame({
            'user_id': np.repeat(user_ids[r], n),
            'item_id': item_ids[top.ravel()],
            'knn_score': top_scores.ravel().astype(np.float32),
        }))
    return (
        pl.concat(out)
        .filter(pl.col('knn_score') > 0)
        .sort(['user_id', 'knn_score'], descending=[False, True])
        .with_columns(knn_rank=pl.int_range(1, pl.len() + 1).over('user_id').cast(pl.Int32))
    )


def build_candidates(log, start):
    hist, warm, _ = split(log, start)
    warm_users = warm['user_id'].unique().sort()
    pop = popular_candidates(hist, warm_users, start)
    knn = knn_candidates(hist, warm_users, start)
    cand = pop.join(knn, on=['user_id', 'item_id'], how='full', coalesce=True)
    labels = warm.select('user_id', 'item_id', target=pl.lit(1, dtype=pl.Int8))
    return (
        cand.join(labels, on=['user_id', 'item_id'], how='left')
        .with_columns(
            pl.col('target').fill_null(0),
            pl.col('pop_rank').fill_null(999), pl.col('knn_rank').fill_null(999),
            pl.col('pop_score').fill_null(0), pl.col('knn_score').fill_null(0),
        )
    )

Сколько кандидатов брать из каждого источника? Больше кандидатов — выше потолок Recall, но таблица признаков растёт линейно. Смотрим покрытие позитивов на `val` в зависимости от размера списка.

In [ ]:
with timer('кривая покрытия'):
    _hist, _warm, _ = split(log, FOLDS['val'])
    _wu = _warm['user_id'].unique().sort()
    _pos = _warm.select('user_id', 'item_id')
    _pop = popular_candidates(_hist, _wu, FOLDS['val'], n=300)
    _knn = knn_candidates(_hist, _wu, FOLDS['val'], n=300)
    rows = []
    for n_pop, n_knn in [(50, 50), (50, 100), (100, 100), (100, 200), (200, 200), (300, 300)]:
        u = pl.concat([
            _pop.filter(pl.col('pop_rank') <= n_pop).select('user_id', 'item_id'),
            _knn.filter(pl.col('knn_rank') <= n_knn).select('user_id', 'item_id'),
        ]).unique()
        rows.append({'популярность': n_pop, 'ItemKNN': n_knn,
                     'кандидатов на пользователя': u.height / _wu.len(),
                     'покрытие позитивов': _pos.join(u, on=['user_id', 'item_id']).height / _pos.height})
del _hist, _warm, _pop, _knn, u
gc.collect()
pl.DataFrame(rows)

Покрытие растёт медленно: даже 500 кандидатов покрывают только 61% позитивов, остальное — длинный хвост, который не дают ни популярность, ни совместные просмотры. Берём 100 + 100. Вариант 100 + 200 добавляет около 4 п.п. покрытия, но увеличивает таблицу признаков в полтора раза, а на 7.7 ГБ RAM при поиске по сотням признаков это критично. Для сравнения ранкеров важен одинаковый набор кандидатов, а не максимальный потолок.

In [ ]:
candidates = {}
for name, start in FOLDS.items():
    with timer(f'кандидаты {name}'):
        candidates[name] = cached(f'candidates_{name}_{N_USERS}_{N_POP}_{N_KNN}', lambda s=start: build_candidates(log, s))

Сколько позитивов покрывают кандидаты. Это верхняя граница Recall для любого ранкера на этих кандидатах.

In [ ]:
rows = []
for name, start in FOLDS.items():
    _, warm, _ = split(log, start)
    c = candidates[name]
    pos = c.filter(pl.col('target') == 1)
    rows.append({
        'фолд': name,
        'кандидатов на пользователя': c.height / c['user_id'].n_unique(),
        'доля позитивов': c['target'].mean(),
        'покрыто позитивов: всего': pos.height / warm.height,
        'популярность': pos.filter(pl.col('pop_rank') < 999).height / warm.height,
        'ItemKNN': pos.filter(pl.col('knn_rank') < 999).height / warm.height,
    })
pl.DataFrame(rows)

Вместе два источника покрывают 44% позитивов, по отдельности 34% и 30%: они заметно дополняют друг друга. Среди кандидатов позитив примерно один на 140, сильный дисбаланс классов. Цифры стабильны между неделями, так что `train`, `val` и `test` сопоставимы.

### 2.4. Метрики

Метрики считаются по каждому тёплому пользователю с хотя бы одним позитивом в неделе и усредняются:

- **Recall@10**: доля его позитивов, попавших в топ-10;
- **NDCG@10**: учитывает позиции попаданий;
- **MAP@10**: средняя точность, нормированная на `min(число позитивов, 10)`.

Пользователь, чьи позитивы не попали в кандидаты, получает нули: метрика честно учитывает потери на первой стадии. Для бинарной классификации пар дополнительно считаем ROC-AUC и LogLoss на кандидатах. Доверительные интервалы получаем бутстрепом по пользователям.

In [ ]:
IDCG = np.cumsum(1 / np.log2(np.arange(2, K + 2)))


def ranking_metrics(scored, target, k=K, per_user=False):
    """scored: user_id, item_id, score. target: user_id, item_id — позитивы недели."""
    top = (
        scored.sort(['user_id', 'score'], descending=[False, True])
        .group_by('user_id', maintain_order=True).head(k)
        .with_columns(rank=pl.int_range(1, pl.len() + 1).over('user_id'))
        .join(target.select('user_id', 'item_id', hit=pl.lit(1)), on=['user_id', 'item_id'], how='left')
        .with_columns(pl.col('hit').fill_null(0))
        .with_columns(
            dcg=pl.col('hit') / (pl.col('rank') + 1).log(2),
            prec=pl.col('hit').cum_sum().over('user_id') / pl.col('rank') * pl.col('hit'),
        )
        .group_by('user_id').agg(hits=pl.col('hit').sum(), dcg=pl.col('dcg').sum(), ap=pl.col('prec').sum())
    )
    users_rel = target.group_by('user_id').len('n_rel')
    pu = (
        users_rel.join(top, on='user_id', how='left').fill_null(0)
        .with_columns(cap=pl.min_horizontal('n_rel', pl.lit(k)))
        .with_columns(
            recall=pl.col('hits') / pl.col('n_rel'),
            ndcg=pl.col('dcg') / pl.col('cap').replace_strict(list(range(1, k + 1)), IDCG[:k].tolist()),
            map=pl.col('ap') / pl.col('cap'),
        )
        .select('user_id', 'recall', 'ndcg', 'map')
    )
    if per_user:
        return pu
    return {f'{m}@{k}': pu[m].mean() for m in ['recall', 'ndcg', 'map']}


def bootstrap_ci(per_user, metric, n_boot=1000, seed=SEED):
    vals = per_user[metric].to_numpy()
    r = np.random.default_rng(seed)
    means = vals[r.integers(0, len(vals), (n_boot, len(vals)))].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

Первые точки отсчёта на `val`: ранжирование кандидатов только по популярности и только по скору ItemKNN. Ранкер должен быть заметно лучше обоих.

In [ ]:
_, warm_val, _ = split(log, FOLDS['val'])
c = candidates['val']
baselines = {
    'популярность': c.select('user_id', 'item_id', score=pl.col('pop_score')),
    'ItemKNN': c.select('user_id', 'item_id', score=pl.col('knn_score')),
}
rows = []
for name, scored in baselines.items():
    pu = ranking_metrics(scored, warm_val, per_user=True)
    lo, hi = bootstrap_ci(pu, 'ndcg')
    rows.append({'модель': name, **{f'{m}@{K}': pu[m].mean() for m in ['recall', 'ndcg', 'map']},
                 'NDCG 95% ДИ': f'[{lo:.4f}, {hi:.4f}]'})
pl.DataFrame(rows)

Популярность обгоняет ItemKNN по всем трём метрикам, доверительные интервалы NDCG не пересекаются. Для KION это ожидаемо: просмотры сконцентрированы на новинках, а у половины пользователей история из 1–2 тайтлов, по которым сходство ненадёжно. При этом Recall@10 популярности 0.20 при потолке кандидатов 0.44: у ранкера есть запас, чтобы комбинировать оба сигнала и добавлять персональные признаки.

## 3. Бейзлайн: LightGBM на ручных признаках

### 3.1. Движок признаков

Почти все ручные признаки в рекомендациях — агрегаты по таблице событий: «сколько раз», «сколько минут», «какая доля досмотров», за какое окно, по какой связи. Поэтому и ручные, и автоматические признаки считает один движок. Каждый признак описывается спецификацией:

| Поле | Значения | Смысл |
|---|---|---|
| `key` | `user`, `item`, `ua:<атрибут>`, `di:<демография>` | по какой сущности или связи агрегируем |
| `event` | `all`, `complete` (≥80%), `drop` (<10%) | какие просмотры учитываем |
| `time` | `all`, `wN` (окно N дней), `hN` (затухание с полупериодом N дней) | как учитываем давность |
| `agg` | `cnt`, `dur`, `pct`, `recency`, `first` | число, суммарное время, средний % просмотра, дней с последнего, дней с первого |
| `norm` | `none`, `rate`, `share`, `lift` | нормировка, см. ниже |
| `reduce` | `max`, `mean`, `sum` | как свернуть многозначный атрибут объекта (у фильма несколько жанров) |

Связи:
- `ua:genre` — пользователь × жанр: насколько пользователь смотрит жанры кандидата. Аналогично страны, режиссёры, актёры (первые пять в титрах: полный список в среднем из 15 имён раздувает таблицу пользователь × актёр до размеров, которые не помещаются в 7.7 ГБ), тип контента, возрастной рейтинг, десятилетие выпуска;
- `di:age` — демографическая группа × объект: насколько объект популярен у пользователей той же возрастной группы. Аналогично доход, пол, наличие детей.

Нормировки:
- `rate` — доля среди всех просмотров той же сущности, например доля досмотров у пользователя;
- `share` — доля внутри родителя: доля жанра в истории пользователя, доля объекта в просмотрах группы;
- `lift` — `share`, делённая на глобальную долю: жанр у пользователя относительно его популярности вообще.

Спецификация компилируется в SQL для DuckDB. Единственный источник событий для признаков фолда — таблица `ev_<fold>`, в которую попадают только события строго до начала целевой недели. Ниже это проверяется отдельным тестом.

In [ ]:
import duckdb
from dataclasses import dataclass

con = duckdb.connect()
con.execute(f"SET threads TO {4 if ON_KAGGLE else 2}")
# на ноутбуке свободно около 2 ГБ, сверх лимита DuckDB выгружает данные на диск; на Kaggle памяти около 30 ГБ
con.execute(f"SET memory_limit = '{'8GB' if ON_KAGGLE else '700MB'}'")
con.execute(f"SET temp_directory = '{(CACHE_DIR / 'duckdb_tmp').as_posix()}'")
con.execute('SET preserve_insertion_order = false')


def explode_attr(col, top=None):
    vals = pl.col(col).str.split(',')
    if top:
        vals = vals.list.head(top)
    return (
        items.select('item_id', val=vals)
        .explode('val')
        .with_columns(pl.col('val').str.strip_chars().str.to_lowercase())
        .filter(pl.col('val').is_not_null() & (pl.col('val') != ''))
        .unique()
    )


ATTRS = {
    'genre': explode_attr('genres'),
    'country': explode_attr('countries'),
    'director': explode_attr('directors'),
    'actor': explode_attr('actors', top=5),       # первые 5 в титрах: главные роли
    'content_type': items.select('item_id', val=pl.col('content_type')),
    'age_rating': items.select('item_id', val=pl.col('age_rating').cast(pl.Int64).cast(pl.Utf8)).drop_nulls(),
    'decade': items.select('item_id', val=(pl.col('release_year') // 10 * 10).cast(pl.Int64).cast(pl.Utf8)).drop_nulls(),
}
for name, df in ATTRS.items():
    df = df.with_columns(nv=pl.len().over('item_id'))
    con.register('tmp', df.to_arrow())
    con.execute(f'CREATE OR REPLACE TABLE attr_{name} AS SELECT * FROM tmp')
    con.unregister('tmp')

DEMOS = ['age', 'income', 'sex', 'kids_flg']
demo = (
    pl.DataFrame({'user_id': sample_users}).join(users, on='user_id', how='left')
    .with_columns([pl.col(c).cast(pl.Utf8).fill_null('unknown') for c in DEMOS])
)
con.register('tmp', demo.to_arrow())
con.execute('CREATE OR REPLACE TABLE demo AS SELECT * FROM tmp')
con.unregister('tmp')

pl.DataFrame({'атрибут': list(ATTRS), 'значений': [d['val'].n_unique() for d in ATTRS.values()],
              'в среднем на объект': [d.height / d['item_id'].n_unique() for d in ATTRS.values()]})

In [ ]:
@dataclass(frozen=True)
class Spec:
    key: str
    event: str = 'all'
    time: str = 'all'
    agg: str = 'cnt'
    norm: str = 'none'
    reduce: str = '-'

    @property
    def name(self):
        return '__'.join([self.key.replace(':', '_'), self.event, self.time, self.agg, self.norm, self.reduce])


EVENT_FILTERS = {'all': 'TRUE', 'complete': 'pct >= 80', 'drop': 'pct < 10'}
AGGS = {
    'cnt': 'SUM(w * {m})',
    'dur': 'SUM(w * {m} * dur)',
    'pct': 'SUM(w * {m} * pct) / NULLIF(SUM(w * {m}), 0)',
    'recency': 'MIN(CASE WHEN {m} > 0 THEN age END)',
    'first': 'MAX(CASE WHEN {m} > 0 THEN age END)',
}
KIND_GROUPS = {          # (группировка, родитель для share, «другая сторона» для lift)
    'user': (['user_id'], None, None),
    'item': (['item_id'], [], None),
    'ua': (['user_id', 'val'], ['user_id'], ['val']),
    'di': (['g', 'item_id'], ['g'], ['item_id']),
}


def is_valid(s):
    kind = s.key.split(':')[0]
    if s.agg in ('recency', 'first') and (s.time != 'all' or s.norm != 'none'):
        return False
    if s.agg == 'pct' and s.norm != 'none':
        return False
    if s.norm == 'rate' and s.event == 'all':
        return False
    if s.norm == 'share' and kind == 'user':
        return False
    if s.norm == 'lift' and kind not in ('ua', 'di'):
        return False
    return (s.reduce != '-') == (kind == 'ua')


def feature_sql(s, fold):
    kind, _, sub = s.key.partition(':')
    w = f'pow(0.5, age / {s.time[1:]})' if s.time.startswith('h') else '1.0'
    tfilt = f'age <= {s.time[1:]}' if s.time.startswith('w') else 'TRUE'
    if s.norm != 'rate' and s.event != 'all':      # знаменатель по всем событиям не нужен
        tfilt += f' AND {EVENT_FILTERS[s.event]}'
    e = (f'SELECT user_id, item_id, dur, pct, age, {w} AS w, '
         f'CAST({EVENT_FILTERS[s.event]} AS DOUBLE) AS hit FROM ev_{fold} WHERE {tfilt}')
    rel = {
        'user': 'e', 'item': 'e',
        'ua': f'e JOIN attr_{sub} USING (item_id)',
        'di': f'(SELECT e.*, d."{sub}" AS g FROM e JOIN demo d USING (user_id))',
    }[kind]
    G, parent, other = KIND_GROUPS[kind]
    A = AGGS[s.agg]

    def agg(cols, m='hit'):
        sel = ', '.join(cols + [A.format(m=m) + ' AS v'])
        return f'SELECT {sel} FROM {rel}' + (f' GROUP BY {", ".join(cols)}' if cols else '')

    ctes = [f'e AS ({e})', f'n AS ({agg(G)})']
    value, joins = 'n.v', ''
    if s.norm == 'rate':
        ctes.append(f'd AS ({agg(G, m="1")})')
        joins += f' JOIN d USING ({", ".join(G)})'
        value = 'n.v / NULLIF(d.v, 0)'
    if s.norm in ('share', 'lift'):
        ctes.append(f'p AS ({agg(parent)})')
        joins += f' JOIN p USING ({", ".join(parent)})' if parent else ' CROSS JOIN p'
        value = 'n.v / NULLIF(p.v, 0)'
    if s.norm == 'lift':
        ctes += [f'o AS ({agg(other)})', f't AS ({agg([])})']
        joins += f' JOIN o USING ({", ".join(other)}) CROSS JOIN t'
        value = f'({value}) / NULLIF(o.v / NULLIF(t.v, 0), 0)'
    ctes.append(f'x AS (SELECT {", ".join("n." + c for c in G)}, {value} AS f FROM n{joins})')

    if kind == 'user':
        body = f'SELECT c.row_id, x.f FROM cand_{fold} c LEFT JOIN x USING (user_id)'
    elif kind == 'item':
        body = f'SELECT c.row_id, x.f FROM cand_{fold} c LEFT JOIN x USING (item_id)'
    elif kind == 'di':
        body = (f'SELECT c.row_id, x.f FROM cand_{fold} c JOIN demo dm USING (user_id) '
                f'LEFT JOIN x ON x.g = dm."{sub}" AND x.item_id = c.item_id')
    else:
        red = {'max': 'MAX(x.f)', 'sum': 'SUM(x.f)', 'mean': 'SUM(x.f) / ANY_VALUE(a.nv)'}[s.reduce]
        ctes.append(
            f'r AS (SELECT c.row_id, {red} AS f FROM cand_{fold} c JOIN attr_{sub} a USING (item_id) '
            f'JOIN x ON x.user_id = c.user_id AND x.val = a.val GROUP BY c.row_id)')
        body = f'SELECT c.row_id, r.f FROM cand_{fold} c LEFT JOIN r USING (row_id)'
    fill = '0' if s.agg in ('cnt', 'dur') and s.norm != 'rate' else 'NULL'
    return f'WITH {", ".join(ctes)} SELECT COALESCE(f, {fill}) AS f FROM ({body}) ORDER BY row_id'


print(feature_sql(Spec('ua:genre', 'complete', 'h30', 'cnt', 'lift', 'max'), 'val'))

Таблицы фолдов. В `ev_<fold>` попадают только события до начала целевой недели, `age` — сколько дней прошло от просмотра до этой даты (от 1 и больше).

В `cand_train` оставляем только пользователей, у которых среди кандидатов есть хотя бы один позитив: остальные не дают градиента для lambdarank. Негативы у них прореживаем до 60 случайных на пользователя, чтобы перебор признаков укладывался в память и время.

Случайность должна быть своей у каждой строки. Первая версия использовала `shuffle(seed).over('user_id')`: перестановка одинакова для всех пользователей, а кандидаты отсортированы по `item_id`, поэтому одни и те же популярные тайтлы никогда не попадали в негативы. Модель выучила их по точному значению `pop_score`, и NDCG@10 на `val` упал до 0.08 при 0.13 у простой популярности. Для `val` и `test` кандидаты полные: метрики считаются на том же списке, который увидел бы пользователь.

In [ ]:
NEG_PER_USER = 60
FEATURE_DIR = CACHE_DIR / f'features_{N_USERS}'


def setup_fold(fold, start, source_log=None):
    src = log if source_log is None else source_log
    con.register('tmp', src.filter(pl.col('dt') < start).to_arrow())
    con.execute(f"""CREATE OR REPLACE TABLE ev_{fold} AS
        SELECT user_id, item_id, total_dur AS dur, watched_pct AS pct,
               date_diff('day', dt, DATE '{start}') AS age
        FROM tmp""")
    con.unregister('tmp')


CAND_HASH = {}


def make_cand_table(name, c):
    c = c.sort(['user_id', 'item_id']).with_row_index('row_id')
    con.register('tmp', c.select('row_id', 'user_id', 'item_id').to_arrow())
    con.execute(f'CREATE OR REPLACE TABLE cand_{name} AS SELECT * FROM tmp')
    con.unregister('tmp')
    # кэш признаков привязан к содержимому таблицы кандидатов, а не только к имени фолда
    CAND_HASH[name] = hex(c.select(pl.struct('user_id', 'item_id').hash(seed=0).sum()).item())[2:12]
    return c


def sample_train(c, neg=NEG_PER_USER, seed=SEED):
    users_pos = c.filter(pl.col('target') == 1)['user_id'].unique()
    c = c.filter(pl.col('user_id').is_in(users_pos.implode()))
    # Случайное число у каждой строки своё. Перестановка через shuffle(seed).over('user_id')
    # одинакова для всех пользователей и выбирает в негативы одни и те же популярные тайтлы.
    rnd = np.random.default_rng(seed).random(c.height)
    return (
        c.with_columns(rnd=pl.Series(rnd))
        .filter((pl.col('target') == 1)
                | (pl.col('rnd').rank('ordinal').over(['user_id', 'target']) <= neg))
        .drop('rnd')
    )


cand = {}
for fold, start in FOLDS.items():
    setup_fold(fold, start)
    c = candidates[fold]
    cand[fold] = make_cand_table(fold, sample_train(c) if fold == 'train' else c)
    n_ev = con.execute(f'SELECT COUNT(*) FROM ev_{fold}').fetchall()[0][0]
    print(f'{fold}: событий в истории {n_ev:,}, строк кандидатов {cand[fold].height:,}, '
          f'пользователей {cand[fold]["user_id"].n_unique():,}, позитивов {cand[fold]["target"].sum():,}')


def compute_features(specs, fold, cache=True):
    """Матрица признаков (строки cand_<fold> × specs), float32, с кэшем на диске."""
    out = np.empty((cand[fold].height, len(specs)), dtype=np.float32)
    d = FEATURE_DIR / f'{fold}_{CAND_HASH[fold]}'
    d.mkdir(parents=True, exist_ok=True)
    for j, s in enumerate(specs):
        path = d / f'{s.name}.npy'
        if cache and path.exists():
            out[:, j] = np.load(path)
            continue
        res = con.execute(feature_sql(s, fold)).fetchnumpy()['f']
        col = np.ma.filled(np.ma.asarray(res).astype(np.float32), np.nan)   # NULL -> NaN
        out[:, j] = col
        if cache:
            np.save(path, col)
    return out

### 3.2. Проверка на утечки

Если движок где-то обращается к будущему, признаки изменятся при удалении будущих событий из лога. Тест: строим историю фолда `val` из лога, в котором **все события начиная с 09.08 удалены**, считаем 40 случайных признаков из всего пространства и сравниваем с обычным расчётом. Совпадение должно быть точным.

Таргет-события имеют дату не раньше начала недели, поэтому в `ev_val` не попадают по построению. Это проверяем по минимальному `age`.

In [ ]:
TIMES = ['all', 'w3', 'w7', 'w14', 'w30', 'w60', 'h3', 'h7', 'h14', 'h30', 'h60']
KEYS = ['user', 'item'] + [f'ua:{a}' for a in ATTRS] + [f'di:{d}' for d in DEMOS]


def full_space():
    specs = []
    for key in KEYS:
        reduces = ['max', 'mean', 'sum'] if key.startswith('ua') else ['-']
        for ev in EVENT_FILTERS:
            for t in TIMES:
                for agg in AGGS:
                    for norm in ['none', 'rate', 'share', 'lift']:
                        for r in reduces:
                            s = Spec(key, ev, t, agg, norm, r)
                            if is_valid(s):
                                specs.append(s)
    return specs


SPACE = full_space()
print(f'спецификаций в полном пространстве: {len(SPACE):,}')

r = np.random.default_rng(SEED)
probe = [SPACE[i] for i in r.choice(len(SPACE), 40, replace=False)]
setup_fold('val_check', FOLDS['val'], source_log=log.filter(pl.col('dt') < FOLDS['val']))
con.execute('CREATE OR REPLACE TABLE cand_val_check AS SELECT * FROM cand_val')
cand['val_check'] = cand['val']
CAND_HASH['val_check'] = CAND_HASH['val']
with timer('40 признаков на val'):
    a = compute_features(probe, 'val', cache=False)
b = compute_features(probe, 'val_check', cache=False)
assert np.array_equal(np.nan_to_num(a, nan=-1), np.nan_to_num(b, nan=-1)), 'признаки зависят от будущих событий'
min_age = con.execute('SELECT MIN(age) FROM ev_val').fetchall()[0][0]
assert min_age >= 1, 'в истории есть события из целевой недели'
print(f'40 признаков совпали точно; минимальный age в истории val: {min_age}')
con.execute('DROP TABLE ev_val_check; DROP TABLE cand_val_check')
del a, b, cand['val_check']
gc.collect()

### 3.3. Базовые признаки

Эти признаки не агрегируют историю и одинаковы для всех подходов: скоры и ранги генераторов кандидатов, статические атрибуты объекта и анкета пользователя. Сравнение ручных и автоматических признаков идёт поверх них.

In [ ]:
AGE_ORD = {f'age_{a}': i for i, a in enumerate(['18_24', '25_34', '35_44', '45_54', '55_64', '65_inf'])}
INC_ORD = {f'income_{a}': i for i, a in enumerate(['0_20', '20_40', '40_60', '60_90', '90_150', '150_inf'])}

user_static = users.select(
    'user_id',
    u_age=pl.col('age').replace_strict(AGE_ORD, default=None),
    u_income=pl.col('income').replace_strict(INC_ORD, default=None),
    u_sex=pl.col('sex').replace_strict({'М': 1, 'Ж': 0}, default=None),
    u_kids=pl.col('kids_flg'),
)
item_static = items.select(
    'item_id',
    i_series=(pl.col('content_type') == 'series').cast(pl.Int8),
    i_year=pl.col('release_year'),
    i_age_rating=pl.col('age_rating'),
    i_for_kids=pl.col('for_kids'),
)
BASE = ['pop_score', 'pop_rank', 'knn_score', 'knn_rank',
        'i_series', 'i_year', 'i_age_rating', 'i_for_kids', 'u_age', 'u_income', 'u_sex', 'u_kids']


def base_matrix(fold):
    c = (
        cand[fold]
        .join(user_static, on='user_id', how='left', maintain_order='left')
        .join(item_static, on='item_id', how='left', maintain_order='left')
        .sort('row_id')
    )
    return c.select(BASE).cast(pl.Float32).to_numpy()


WARM = {fold: split(log, start)[1] for fold, start in FOLDS.items()}

### 3.4. Ручной набор признаков

Набор, который я бы собрал вручную, не перебирая варианты: стандартные окна 7 и 30 дней и вся история, по одному-два признака на каждую связь.

In [ ]:
MANUAL = [
    # пользователь: активность, глубина просмотра, давность
    Spec('user', 'all', 'all', 'cnt'), Spec('user', 'all', 'w7', 'cnt'), Spec('user', 'all', 'w30', 'cnt'),
    Spec('user', 'all', 'all', 'dur'), Spec('user', 'all', 'all', 'pct'),
    Spec('user', 'complete', 'all', 'cnt', 'rate'),
    Spec('user', 'all', 'all', 'recency'), Spec('user', 'all', 'all', 'first'),
    # объект: популярность, тренд, досматриваемость, новизна
    Spec('item', 'all', 'w7', 'cnt'), Spec('item', 'all', 'w30', 'cnt'), Spec('item', 'all', 'all', 'cnt'),
    Spec('item', 'all', 'w7', 'cnt', 'share'), Spec('item', 'all', 'w30', 'pct'),
    Spec('item', 'complete', 'w30', 'cnt', 'rate'),
    Spec('item', 'all', 'all', 'first'), Spec('item', 'all', 'all', 'recency'),
    # пользователь × атрибуты объекта
    Spec('ua:genre', 'all', 'all', 'cnt', 'share', 'mean'), Spec('ua:genre', 'all', 'w30', 'cnt', 'share', 'max'),
    Spec('ua:genre', 'complete', 'all', 'cnt', 'share', 'max'),
    Spec('ua:country', 'all', 'all', 'cnt', 'share', 'max'),
    Spec('ua:director', 'all', 'all', 'cnt', 'none', 'max'),
    Spec('ua:actor', 'all', 'all', 'cnt', 'none', 'sum'),
    Spec('ua:content_type', 'all', 'all', 'cnt', 'share', 'max'),
    Spec('ua:age_rating', 'all', 'all', 'cnt', 'share', 'max'),
    Spec('ua:decade', 'all', 'all', 'cnt', 'share', 'max'),
    # демографическая группа × объект
    Spec('di:age', 'all', 'w30', 'cnt', 'lift'), Spec('di:sex', 'all', 'w30', 'cnt', 'lift'),
    Spec('di:kids_flg', 'all', 'w30', 'cnt', 'lift'), Spec('di:income', 'all', 'w30', 'cnt', 'lift'),
]
assert all(is_valid(s) for s in MANUAL)

with timer('ручные признаки, train + val'):
    XB = {f: base_matrix(f) for f in ['train', 'val']}
    XM = {f: compute_features(MANUAL, f) for f in ['train', 'val']}
print(f'{len(BASE)} базовых + {len(MANUAL)} ручных признаков')

### 3.5. Обучение и оценка

LightGBM с `lambdarank`: пользователь — это группа, внутри неё модель учится ставить позитивы выше негативов. Early stopping по NDCG@10 на `val`. Гиперпараметры одинаковы для всех подходов; `num_leaves` выбираем на ручном наборе ниже.

LogLoss не считаем: у `lambdarank` скоры не являются вероятностями. ROC-AUC считаем по всем парам кандидатов `val`, он показывает качество разделения позитивов и негативов независимо от позиций в топе.

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

LGB_PARAMS = dict(
    objective='lambdarank', metric='ndcg', eval_at=[K], learning_rate=0.05, num_leaves=31,
    min_data_in_leaf=50, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
    lambdarank_truncation_level=20, verbose=-1, seed=SEED, num_threads=4,
)


def groups(fold):
    return cand[fold].group_by('user_id', maintain_order=True).len()['len'].to_numpy()


def fit_ranker(Xtr, Xva, names, params=None, rounds=2000, es=50, tr='train', va='val'):
    params = {**LGB_PARAMS, **(params or {})}
    dtr = lgb.Dataset(Xtr, cand[tr]['target'].to_numpy(), group=groups(tr), feature_name=names, free_raw_data=True)
    dva = lgb.Dataset(Xva, cand[va]['target'].to_numpy(), group=groups(va), reference=dtr)
    return lgb.train(params, dtr, rounds, valid_sets=[dva], callbacks=[lgb.early_stopping(es, verbose=False)])


def evaluate(scores, fold, per_user=False):
    scored = cand[fold].select('user_id', 'item_id', score=pl.Series(scores))
    pu = ranking_metrics(scored, WARM[fold], per_user=True)
    res = {f'{m}@{K}': pu[m].mean() for m in ['recall', 'ndcg', 'map']}
    res['auc'] = roc_auc_score(cand[fold]['target'].to_numpy(), scores)
    return (res, pu) if per_user else res


def run(name, Xtr, Xva, names, params=None, tr='train', va='val'):
    t0 = time.perf_counter()
    model = fit_ranker(Xtr, Xva, names, params, tr=tr, va=va)
    fit_time = time.perf_counter() - t0
    res = evaluate(model.predict(Xva), va)
    return {'модель': name, 'признаков': len(names), **res, 'деревьев': model.best_iteration, 'обучение, с': fit_time}, model


names_manual = BASE + [s.name for s in MANUAL]
X_manual = {f: np.hstack([XB[f], XM[f]]) for f in ['train', 'val']}

rows = []
for leaves in [15, 31, 63, 127]:
    r, _ = run(f'num_leaves={leaves}', X_manual['train'], X_manual['val'], names_manual, {'num_leaves': leaves})
    rows.append(r)
pl.DataFrame(rows)

Все варианты укладываются в 0.0015 NDCG, это меньше ширины доверительного интервала (около 0.009). Берём лучшее значение и фиксируем его для всех следующих экспериментов, чтобы сравнение наборов признаков не смешивалось с подбором модели.

In [ ]:
best_leaves = int(max(rows, key=lambda r: r[f'ndcg@{K}'])['модель'].split('=')[1])
LGB_PARAMS['num_leaves'] = best_leaves
print('num_leaves =', best_leaves)

results = []
r, _ = run('только базовые', XB['train'], XB['val'], BASE)
results.append(r)
r, model_manual = run('базовые + ручные', X_manual['train'], X_manual['val'], names_manual)
results.append(r)
pl.DataFrame(results)

Ранкер только на базовых признаках уже лучше популярности (NDCG@10 0.146 против 0.131): он комбинирует два генератора и учитывает возрастной рейтинг и анкету. Ручные агрегаты добавляют ещё 0.011 NDCG (+8%) и 0.01 Recall@10. Это и есть планка, с которой сравниваем автоматическую генерацию.

### 3.6. Абляция по группам ручных признаков

Убираем по одной группе признаков и смотрим, сколько теряет модель. Это показывает, откуда берётся прирост над базовыми признаками.

In [ ]:
GROUPS = {
    'пользователь': lambda s: s.key == 'user',
    'объект': lambda s: s.key == 'item',
    'пользователь × атрибуты': lambda s: s.key.startswith('ua:'),
    'демография × объект': lambda s: s.key.startswith('di:'),
}
ablation = []
for gname, pred in GROUPS.items():
    keep = [j for j, s in enumerate(MANUAL) if not pred(s)]
    names = BASE + [MANUAL[j].name for j in keep]
    Xtr = np.hstack([XB['train'], XM['train'][:, keep]])
    Xva = np.hstack([XB['val'], XM['val'][:, keep]])
    r, _ = run(f'без группы «{gname}»', Xtr, Xva, names)
    ablation.append(r)
ref = results[1][f'ndcg@{K}']
pl.DataFrame(ablation).with_columns((pl.col(f'ndcg@{K}') - ref).alias('Δ NDCG'))

Каждая группа по отдельности даёт немного: без любой из них NDCG падает на 0.0002–0.0027. Группы частично дублируют друг друга, например популярность объекта за 7 дней почти совпадает со скором генератора популярности (корреляция 0.998). Больше всего теряется без связей «пользователь × атрибуты» (−0.0027): это единственный источник персонального сигнала, кроме ItemKNN. Почти столько же без признаков объекта (−0.0022): популярность за разные окна и досматриваемость дополняют скор генератора популярности. Признаки пользователя почти бесполезны, и это ожидаемо: они одинаковы для всех кандидатов одного пользователя и могут влиять на порядок только через взаимодействия с другими признаками.

In [ ]:
imp = pl.DataFrame({'признак': names_manual, 'gain': model_manual.feature_importance('gain')})
imp = imp.with_columns(pl.col('gain') / pl.col('gain').sum()).sort('gain', descending=True).head(20)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp['признак'].to_list()[::-1], imp['gain'].to_numpy()[::-1], color=COLORS[0], height=0.7)
ax.set_title('Ручной набор: доля gain, топ-20 признаков', loc='left')
ax.grid(axis='y', visible=False)
fig.tight_layout()
plt.show()

Наверху скоры генераторов и популярность объекта, дальше связи пользователя с жанрами, странами и типом контента. Признаки пользователя в топе почти не встречаются, это согласуется с абляцией.

## 4. Автоматическая генерация признаков

Движок из раздела 3 задаёт пространство из 7 389 допустимых спецификаций: 13 связей, 3 типа событий, 11 вариантов учёта времени (окна от 3 до 60 дней и затухание с полупериодом от 3 до 60 дней), 5 агрегаций, 4 нормировки, 3 свёртки. Ручной набор — 29 точек этого пространства. Вопрос: найдёт ли алгоритм набор не хуже, если человек задаёт только пространство, но не выбирает точки?

Сравниваем две стратегии:

1. **Сетка + фильтр** (по мотивам Deep Feature Synthesis). Берём фиксированную сетку стандартных примитивов, считаем все признаки, отбираем лучшие по одномерному критерию с удалением дублей. Дёшево, но признаки отбираются по отдельности, без учёта того, что уже есть в модели.
2. **Жадный forward-отбор с успешным отсевом.** На каждом шаге генерируем пул кандидатов из полного пространства, дёшево оцениваем каждый в составе текущего набора, лучших проверяем полной моделью и добавляем один, если он улучшает NDCG на `val`.

В разделе 5 LLM-агент заменяет генератор пула во второй стратегии, всё остальное остаётся тем же.

### 4.1. Одномерный критерий: AUC внутри пользователя

Обычный AUC по всем парам вознаграждает признаки, которые отличают активных пользователей от неактивных, хотя для ранжирования это бесполезно: порядок кандидатов внутри пользователя от них не меняется. Поэтому считаем AUC отдельно для каждого пользователя и усредняем. У признака, постоянного внутри пользователя, такой AUC ровно 0.5.

In [ ]:
del X_manual, XM
gc.collect()


def within_user_auc(x, fold='train'):
    c = cand[fold].select('user_id', 'target').with_columns(x=pl.Series(np.nan_to_num(x, nan=-1e12)))
    g = (
        c.with_columns(rk=pl.col('x').rank('average').over('user_id'))
        .group_by('user_id')
        .agg(npos=pl.col('target').sum(), n=pl.len(), rs=(pl.col('rk') * pl.col('target')).sum())
        .filter((pl.col('npos') > 0) & (pl.col('npos') < pl.col('n')))
    )
    auc = (g['rs'] - g['npos'] * (g['npos'] + 1) / 2) / (g['npos'] * (g['n'] - g['npos']))
    return float(auc.mean())


Xm_train = compute_features(MANUAL, 'train')
uauc = pl.DataFrame({
    'признак': [s.name for s in MANUAL],
    'AUC внутри пользователя': [within_user_auc(Xm_train[:, j]) for j in range(len(MANUAL))],
}).sort('AUC внутри пользователя', descending=True)
del Xm_train
uauc.head(8)

Признаки пользователя дают ровно 0.5, как и должно быть. Лучшие одномерные сигналы — популярность объекта (0.77) и её вариант внутри демографической группы (0.59). Из персональных связей лучше всего доля стран кандидата в истории пользователя (0.57).

### 4.2. Стратегия 1: сетка + фильтр

Сетка повторяет то, что сделал бы DFS со стандартными примитивами: окна 7 и 30 дней и вся история, все просмотры и досмотры, все агрегации и нормировки, свёртки `max` и `mean`. Отбор:

1. считаем все признаки сетки на `train` и сортируем по |AUC внутри пользователя − 0.5|;
2. идём по списку и пропускаем признак, если его корреляция с уже отобранным больше 0.95;
3. обучаем модели на топ-k отобранных при k = 10, 20, 30, 45.

In [ ]:
DFS_GRID = [s for s in SPACE
            if s.time in ('all', 'w7', 'w30') and s.event in ('all', 'complete') and s.reduce in ('-', 'max', 'mean')]
print(f'признаков в сетке: {len(DFS_GRID)}')

with timer('сетка: расчёт на train и одномерный отбор'):
    dfs_scores = []
    for s in DFS_GRID:
        x = compute_features([s], 'train')[:, 0]
        dfs_scores.append(abs(within_user_auc(x) - 0.5))
    order = np.argsort(dfs_scores)[::-1]

    selected, kept_cols = [], []
    for j in order:
        if len(selected) == 45:
            break
        x = compute_features([DFS_GRID[j]], 'train')[:, 0]
        x = np.nan_to_num(x, nan=np.nanmin(x) - 1 if np.isfinite(np.nanmin(x)) else 0)
        if x.std() == 0:
            continue
        if any(abs(np.corrcoef(x, k)[0, 1]) > 0.95 for k in kept_cols):
            continue
        selected.append(DFS_GRID[j])
        kept_cols.append(x)
    del kept_cols
DFS_SELECTED = selected
print('первые 10 отобранных:')
for s in DFS_SELECTED[:10]:
    print(' ', s.name)

In [ ]:
def matrices(specs, folds=('train', 'val')):
    return {f: np.hstack([XB[f], compute_features(specs, f)]) if specs else XB[f] for f in folds}


with timer('сетка: обучение моделей'):
    dfs_rows = []
    for k in [10, 20, 30, 45]:
        specs = DFS_SELECTED[:k]
        X = matrices(specs)
        r, _ = run(f'сетка, топ-{k}', X['train'], X['val'], BASE + [s.name for s in specs])
        dfs_rows.append(r)
        del X
        gc.collect()
pl.DataFrame(dfs_rows)

Сетка с фильтром проигрывает ручному набору: лучший вариант (топ-30) даёт 0.1527 против 0.1568, при этом размер отобранного набора почти не влияет на качество. Причина видна в списке отобранных признаков: одномерный критерий ставит наверх популярность объекта в разных демографических группах и окнах. По отдельности каждый такой признак сильный, но порог корреляции 0.95 их не отсекает, и вместе они почти не добавляют информации сверх скора генератора популярности. Персональные связи с более слабым одномерным AUC до отбора не доходят. Это известная слабость DFS-подхода: признаки отбираются независимо друг от друга.

### 4.3. Стратегия 2: жадный forward-отбор с успешным отсевом

Один шаг поиска:

1. **Пул.** Генератор предлагает кандидатов из полного пространства. Для случайного поиска: 120 случайных спецификаций, из которых оставляем 24 с лучшим AUC внутри пользователя.
2. **Дешёвая оценка.** Каждый из 24 кандидатов добавляется к текущему набору, модель с `learning_rate=0.15` обучается на `train` и оценивается по NDCG@10 на подвыборке 3 000 пользователей `val`.
3. **Полная оценка.** Четыре лучших кандидата проверяются полной моделью на всём `val`.
4. **Принятие.** Лучший добавляется в набор, если улучшает NDCG@10 больше чем на 0.0003. Иначе шаг пропадает.

12 шагов, всего 288 дешёвых и 48 полных обучений. Этот бюджет одинаков для всех генераторов пула. Результаты поиска кэшируются в `cache/search_<имя>.json`.

In [ ]:
import json

STEPS, POOL, TOP, SCREEN, EPS = 12, 24, 4, 120, 0.0003
CHEAP_PARAMS = {'learning_rate': 0.15}

rng_sub = np.random.default_rng(SEED)
val_users = cand['val']['user_id'].unique().sort().to_numpy()
VAL_SUB = np.isin(cand['val']['user_id'].to_numpy(), rng_sub.choice(val_users, 3000, replace=False))
groups_val_sub = cand['val'].filter(pl.Series(VAL_SUB)).group_by('user_id', maintain_order=True).len()['len'].to_numpy()


def cheap_score(specs):
    X = matrices(specs)
    names = BASE + [s.name for s in specs]
    dtr = lgb.Dataset(X['train'], cand['train']['target'].to_numpy(), group=groups('train'), feature_name=names)
    dva = lgb.Dataset(X['val'][VAL_SUB], cand['val']['target'].to_numpy()[VAL_SUB], group=groups_val_sub, reference=dtr)
    m = lgb.train({**LGB_PARAMS, **CHEAP_PARAMS}, dtr, 300, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(20, verbose=False)])
    return m.best_score['valid_0'][f'ndcg@{K}']


def full_score(specs):
    X = matrices(specs)
    r, _ = run('', X['train'], X['val'], BASE + [s.name for s in specs])
    return r[f'ndcg@{K}']


def spec_from_name(name):
    key, *rest = name.split('__')
    for prefix in ('ua_', 'di_'):
        if key.startswith(prefix):
            key = prefix[:2] + ':' + key[3:]
    return Spec(key, *rest)


def greedy_search(name, propose):
    """propose(current, history) -> список кандидатов (Spec)."""
    path = CACHE_DIR / f'search_{name}.json'
    if path.exists():
        saved = json.loads(path.read_text(encoding='utf-8'))
        return [spec_from_name(n) for n in saved['accepted']], saved['history']
    current, history = [], []
    best = full_score(current)
    t0 = time.perf_counter()
    for step in range(STEPS):
        pool = [s for s in dict.fromkeys(propose(current, history)) if s not in current][:POOL]
        cheap = sorted(((cheap_score(current + [s]), s) for s in pool), key=lambda t: -t[0])
        full = [(full_score(current + [s]), s) for _, s in cheap[:TOP]]
        score, s_best = max(full, key=lambda t: t[0])
        accepted = score > best + EPS
        if accepted:
            current.append(s_best)
            best = score
        history.append({
            'step': step, 'accepted': s_best.name if accepted else None, 'ndcg': best,
            'tried': [{'spec': s.name, 'cheap': c} for c, s in cheap],
            'full': [{'spec': s.name, 'ndcg': f} for f, s in full],
            'elapsed': time.perf_counter() - t0,
        })
        print(f'шаг {step:2d}: NDCG {best:.4f}  {"+ " + s_best.name if accepted else "без изменений"}', flush=True)
    path.write_text(json.dumps({'accepted': [s.name for s in current], 'history': history},
                               ensure_ascii=False, indent=1), encoding='utf-8')
    return current, history


rng_search = np.random.default_rng(SEED)


def propose_random(current, history):
    idx = rng_search.choice(len(SPACE), SCREEN, replace=False)
    specs = [SPACE[i] for i in idx if SPACE[i] not in current]
    scores = [abs(within_user_auc(compute_features([s], 'train')[:, 0]) - 0.5) for s in specs]
    return [specs[i] for i in np.argsort(scores)[::-1][:POOL]]


with timer('жадный поиск, случайный пул'):
    GREEDY, greedy_hist = greedy_search('greedy_random', propose_random)
for s in GREEDY:
    print(' ', s.name)

Жадный поиск со случайным пулом за 12 шагов принял 6 признаков и дошёл до NDCG@10 0.1562, почти до ручного набора (0.1568), используя в пять раз меньше признаков. Поиск занял около 40 минут. Что он выбрал:

- досматриваемость и брошенные просмотры объекта (`item drop w30 dur share`, `item all dur`): признаки качества тайтла, а не только популярности;
- связи пользователя с жанрами, странами и десятилетием выпуска, в том числе с затуханием `h7` и окном `w60` — значения, которых нет в ручном наборе;
- популярность у пользователей того же пола за неделю.

Половина шагов (6 из 12) прошла без принятия: в случайном пуле не нашлось кандидата, который улучшил бы набор больше чем на 0.0003.

## 5. LLM-агент как генератор пула

Случайный генератор ничего не знает о предметной области и о том, что уже сработало. LLM знает и то и другое: ей можно описать данные, грамматику и историю поиска. Проверяем, даст ли это лучший пул кандидатов при том же бюджете.

Устройство:

- модель `nvidia/nemotron-3-ultra-550b-a55b:free` через OpenRouter (бесплатная, 550B параметров, режим рассуждений);
- на каждом шаге агент получает описание данных и грамматики, текущий набор признаков с траекторией NDCG и результаты предыдущих шагов: какие кандидаты пробовались, их дешёвая и полная оценки;
- агент отвечает JSON-списком из 24 спецификаций. Код проверяет каждую на допустимость, недопустимые отбрасывает. Если допустимых меньше половины, агенту один раз возвращается список ошибок;
- дальше всё как в разделе 4.3: дешёвая оценка, полная оценка четырёх лучших, принятие по порогу.

Агент не пишет SQL и не видит данных напрямую, только агрегированные результаты на `train` и `val`. Утечка из будущего невозможна: признаки считает тот же движок. Агент не видит и ручной набор из раздела 3, иначе сравнение потеряло бы смысл.

Все запросы и ответы сохраняются в `cache/llm_log.jsonl`. При перезапуске ноутбук берёт ответы оттуда, поэтому ключ API нужен только при первом прогоне.

In [ ]:
import hashlib
import re

from dotenv import load_dotenv
from openai import OpenAI

LLM_MODEL = 'nvidia/nemotron-3-ultra-550b-a55b:free'
LLM_LOG = CACHE_DIR / 'llm_log.jsonl'
load_dotenv('.env')

llm_cache = {}
if LLM_LOG.exists():
    for line in LLM_LOG.read_text(encoding='utf-8').splitlines():
        rec = json.loads(line)
        llm_cache[rec['hash']] = rec
LLM_STATS = {'calls': 0, 'cached': 0, 'seconds': 0.0, 'prompt_tokens': 0, 'completion_tokens': 0}


def llm(messages, temperature=0.7):
    h = hashlib.sha256(json.dumps([LLM_MODEL, messages, temperature], ensure_ascii=False).encode()).hexdigest()
    if h in llm_cache:
        LLM_STATS['cached'] += 1
        return llm_cache[h]['response']
    if 'OPENROUTER_API_KEY' not in os.environ and ON_KAGGLE:      # на Kaggle ключ хранится в Add-ons → Secrets
        from kaggle_secrets import UserSecretsClient
        os.environ['OPENROUTER_API_KEY'] = UserSecretsClient().get_secret('OPENROUTER_API_KEY')
    client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=os.environ['OPENROUTER_API_KEY'])
    for attempt in range(6):
        try:
            t0 = time.perf_counter()
            r = client.chat.completions.create(model=LLM_MODEL, messages=messages,
                                               temperature=temperature, max_tokens=12000)
            text = r.choices[0].message.content or ''
            if not text.strip():
                raise RuntimeError('пустой ответ')
            break
        except Exception as e:                       # лимиты бесплатного тарифа: ждём и повторяем
            print(f'  запрос не удался ({type(e).__name__}: {str(e)[:120]}), повтор через {30 * (attempt + 1)} с')
            time.sleep(30 * (attempt + 1))
    else:
        raise RuntimeError('LLM недоступна')
    LLM_STATS['calls'] += 1
    LLM_STATS['seconds'] += time.perf_counter() - t0
    if r.usage:
        LLM_STATS['prompt_tokens'] += r.usage.prompt_tokens
        LLM_STATS['completion_tokens'] += r.usage.completion_tokens
    rec = {'hash': h, 'model': LLM_MODEL, 'messages': messages, 'response': text}
    with LLM_LOG.open('a', encoding='utf-8') as f:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    llm_cache[h] = rec
    return text

Промпт на английском: модели обучены в основном на нём, и формат ответа соблюдается надёжнее. Описание грамматики — та же таблица, что в разделе 3.1, плюс правила допустимости из `is_valid`.

In [ ]:
SYSTEM_PROMPT = f"""You are a feature engineer for a recommender system ranking model (LightGBM lambdarank).

Data: an online cinema (KION). Event log: one row per (user, item) with the date of the LAST watch,
total watch duration and watched_pct (0-100). Repeat watches are invisible. Items have genres, countries,
directors, actors, content_type (film/series), age_rating, release decade. Users have age, income, sex,
kids_flg. Task: for each user, rank ~170 candidate items (from a popularity and an ItemKNN generator) by the
chance the user watches them during the next 7 days. Items already watched are never candidates.
The model already has base features: popularity and ItemKNN scores/ranks, item static attributes, user demographics.

You add aggregate features described by a grammar. A feature is a JSON object:
- "key": "user" | "item" | "ua:<attr>" | "di:<demo>"
    user = aggregate over the user's history; item = over the item's history (all users);
    ua:<attr> = user x attribute of the candidate item, attr in {list(ATTRS)};
    di:<demo> = demographic group of the user x candidate item, demo in {DEMOS}.
- "event": "all" | "complete" (watched_pct >= 80) | "drop" (watched_pct < 10)
- "time": one of {TIMES}. "wN" = only events in the last N days; "hN" = all events with exponential
    decay, half-life N days; "all" = whole history, no weighting.
- "agg": "cnt" (weighted count) | "dur" (weighted total duration) | "pct" (weighted mean watched_pct) |
    "recency" (days since the last matching event) | "first" (days since the first matching event)
- "norm": "none" | "rate" (value / same value over all events of the same entity) |
    "share" (value / value of the parent: ua -> the user's total, di -> the group's total, item -> all items) |
    "lift" (share / global share of the attribute value or item)
- "reduce": for ua keys only, how to combine several attribute values of the item: "max" | "mean" | "sum";
    for all other keys it must be "-".

Validity rules: recency/first require time="all" and norm="none"; pct requires norm="none";
rate requires event != "all"; share is not allowed for key "user"; lift only for ua and di keys.

Selection works greedily: each step, your candidates are each added to the current set and evaluated
(cheap model, then the best 4 with a full model, NDCG@10 on validation); the best one is accepted if it
improves NDCG@10 by more than {EPS}. Features that are constant within a user (key "user") cannot change
the ranking of one user's candidates on their own. Propose features that add NEW information beyond what is
already in the set; near-duplicates of accepted features rarely help."""


def describe_history(current, history, last=4):
    lines = [f'Current set ({len(current)} features): ' + (', '.join(s.name for s in current) or 'empty, base features only')]
    if history:
        lines.append('NDCG@10 after each step: ' + ', '.join(f"{h['ndcg']:.4f}" for h in history))
    for h in history[-last:]:
        lines.append(f"\nStep {h['step']}: accepted = {h['accepted'] or 'nothing'}")
        tried = h['tried']
        lines.append('  best cheap scores: ' + '; '.join(f"{t['spec']} {t['cheap']:.4f}" for t in tried[:6]))
        lines.append('  worst cheap scores: ' + '; '.join(f"{t['spec']} {t['cheap']:.4f}" for t in tried[-3:]))
        lines.append('  full model: ' + '; '.join(f"{t['spec']} {t['ndcg']:.4f}" for t in h['full']))
    tried_all = sorted({t['spec'] for h in history for t in h['tried']})
    if tried_all:
        lines.append(f'\nAlready tried, do not repeat ({len(tried_all)}): ' + ', '.join(tried_all))
    return '\n'.join(lines)


FIELDS = ['key', 'event', 'time', 'agg', 'norm', 'reduce']
LLM_PARSE = {'proposed': 0, 'valid': 0, 'duplicates': 0}


def parse_specs(text):
    m = re.search(r'\[.*\]', text, flags=re.S)
    if not m:
        return [], ['no JSON list found']
    try:
        raw = json.loads(m.group(0))
    except json.JSONDecodeError as e:
        return [], [f'invalid JSON: {e}']
    specs, errors = [], []
    space = set(SPACE)
    for obj in raw:
        try:
            s = Spec(*[str(obj.get(f, '-' if f == 'reduce' else '')) for f in FIELDS])
        except Exception:
            errors.append(f'bad object {obj}')
            continue
        if s in space:
            specs.append(s)
        else:
            errors.append(f'{s.name}: not in the grammar or breaks a validity rule')
    return specs, errors


def propose_llm(current, history):
    user_msg = (describe_history(current, history)
                + f'\n\nPropose {POOL} new candidate features. Think about what signal is still missing. '
                  f'Answer with a JSON list of {POOL} objects with keys {FIELDS}, nothing else after the list.')
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_msg}]
    text = llm(messages)
    specs, errors = parse_specs(text)
    if len(specs) < POOL // 2:
        messages += [{'role': 'assistant', 'content': text},
                     {'role': 'user', 'content': 'Some features were invalid:\n' + '\n'.join(errors[:20])
                      + f'\nReturn a corrected JSON list of {POOL} valid features.'}]
        text = llm(messages)
        specs2, errors = parse_specs(text)
        specs += specs2
    tried = {t['spec'] for h in history for t in h['tried']}
    LLM_PARSE['proposed'] += len(specs) + len(errors)
    LLM_PARSE['valid'] += len(specs)
    fresh = [s for s in dict.fromkeys(specs) if s.name not in tried and s not in current]
    LLM_PARSE['duplicates'] += len(specs) - len(fresh)
    return fresh


with timer('жадный поиск, пул от LLM'):
    GREEDY_LLM, llm_hist = greedy_search('greedy_llm', propose_llm)
stats_path = CACHE_DIR / 'llm_stats.json'
if LLM_PARSE['proposed']:
    stats_path.write_text(json.dumps({'llm': LLM_STATS, 'parse': LLM_PARSE}), encoding='utf-8')
elif stats_path.exists():                             # поиск взят из кэша: статистика первого прогона
    saved = json.loads(stats_path.read_text(encoding='utf-8'))
    LLM_STATS, LLM_PARSE = saved['llm'], saved['parse']
for s in GREEDY_LLM:
    print(' ', s.name)
print(f"запросов к LLM: {LLM_STATS['calls']}, время ответов: {LLM_STATS['seconds'] / 60:.1f} мин, "
      f"токенов: {LLM_STATS['prompt_tokens']:,} на входе, {LLM_STATS['completion_tokens']:,} на выходе")
print(f"предложено спецификаций: {LLM_PARSE['proposed']}, допустимых: {LLM_PARSE['valid']} "
      f"({LLM_PARSE['valid'] / max(LLM_PARSE['proposed'], 1):.0%}), повторов уже испробованных: {LLM_PARSE['duplicates']}")

### 5.1. Чем пул LLM отличается от случайного

Сравниваем, какие кандидаты попали в дешёвую оценку у двух генераторов: по учёту времени и по типу связи.

In [ ]:
def pool_profile(hist, field):
    pos = {'key': 0, 'time': 2}[field]
    vals = [t['spec'].split('__')[pos].split('_')[0] if field == 'key' else t['spec'].split('__')[pos]
            for h in hist for t in h['tried']]
    return pl.DataFrame({field: vals}).group_by(field).len().with_columns(pl.col('len') / len(vals))


for field in ['time', 'key']:
    prof = (pool_profile(greedy_hist, field).rename({'len': 'случайный пул'})
            .join(pool_profile(llm_hist, field).rename({'len': 'пул от LLM'}), on=field, how='full', coalesce=True)
            .fill_null(0).sort('пул от LLM', descending=True))
    print(prof)
print('кандидатов на шаг у LLM:', [len(h['tried']) for h in llm_hist])

Три отличия.

1. **Узкий охват времени.** 42% кандидатов LLM используют масштаб 14 дней (`h14` или `w14`), а из шести принятых признаков четыре — `h14`. Случайный генератор покрывает все 11 вариантов почти равномерно. Модель предлагает «разумные» значения, но не исследует пространство.
2. **Признаки пользователя.** 15% предложений — агрегаты уровня пользователя, хотя в промпте сказано, что сами по себе они не меняют порядок кандидатов одного пользователя. В случайном пуле их нет: одномерный отбор по AUC внутри пользователя отсекает их автоматически.
3. **Неполный пул.** На четырёх шагах LLM вернула 4–10 новых кандидатов вместо 24: часть ответа повторяла уже испробованное или ломала правила грамматики. Всего LLM-поиск оценил 211 кандидатов против 288 у случайного, то есть фактически работал с меньшим бюджетом.

Допустимых спецификаций 89%: формат грамматики модель освоила хорошо. Ограничение не в формате, а в разнообразии предложений.

## 6. Сравнение на тестовой неделе

Протокол одинаков для всех подходов:

1. Набор признаков и число деревьев фиксируются по `train → val`, так же, как в поиске.
2. Модель переобучается на неделе `val` (история до 09.08, таргет 09.08–15.08, негативы прорежены так же) с этим числом деревьев.
3. Предсказываем кандидатов недели 16.08–22.08. Тест до этого момента не использовался ни разу.

Для каждого подхода считаем доверительный интервал NDCG@10 бутстрепом по пользователям и парную разность с ручным набором: пользователи одни и те же, поэтому парный бутстреп заметно точнее, чем сравнение двух отдельных интервалов.

In [ ]:
setup_fold('valtr', FOLDS['val'])
cand['valtr'] = make_cand_table('valtr', sample_train(candidates['val']))
XB['valtr'] = base_matrix('valtr')
XB['test'] = base_matrix('test')

best_dfs = max(dfs_rows, key=lambda r: r[f'ndcg@{K}'])
DFS_BEST = DFS_SELECTED[:int(best_dfs['модель'].split('-')[1])]

APPROACHES = {
    'только базовые': [],
    'ручные': MANUAL,
    f'сетка + фильтр (топ-{len(DFS_BEST)})': DFS_BEST,
    'жадный, случайный пул': GREEDY,
    'жадный, пул от LLM': GREEDY_LLM,
    'ручные + жадный (LLM)': MANUAL + [s for s in GREEDY_LLM if s not in MANUAL],
}


def final_eval(name, specs, folds=('train', 'val', 'valtr', 'test')):
    tr, va, refit, te = folds
    names = BASE + [s.name for s in specs]
    X = matrices(specs, folds=(tr, va))
    r_val, model = run(name, X[tr], X[va], names, tr=tr, va=va)
    del X
    gc.collect()
    Xf = matrices(specs, folds=(refit, te))
    dtr = lgb.Dataset(Xf[refit], cand[refit]['target'].to_numpy(), group=groups(refit), feature_name=names)
    m = lgb.train(LGB_PARAMS, dtr, model.best_iteration)
    res, pu = evaluate(m.predict(Xf[te]), te, per_user=True)
    del Xf
    gc.collect()
    return {'подход': name, 'признаков': len(names), 'NDCG val': r_val[f'ndcg@{K}'],
            **{f'{k} test': v for k, v in res.items()}}, pu.sort('user_id')


with timer('финальная оценка'):
    final_rows, per_user = [], {}
    pop_res, pop_pu = evaluate(XB['test'][:, BASE.index('pop_score')], 'test', per_user=True)
    final_rows.append({'подход': 'популярность', 'признаков': 1, 'NDCG val': None,
                       **{f'{k} test': v for k, v in pop_res.items()}})
    per_user['популярность'] = pop_pu.sort('user_id')
    for name, specs in APPROACHES.items():
        row, pu = final_eval(name, specs)
        final_rows.append(row)
        per_user[name] = pu


def paired_ci(a, b, n_boot=2000, seed=SEED):
    d = a - b
    r = np.random.default_rng(seed)
    means = d[r.integers(0, len(d), (n_boot, len(d)))].mean(axis=1)
    return d.mean(), *np.percentile(means, [2.5, 97.5])


ref = per_user['ручные']['ndcg'].to_numpy()
for row in final_rows:
    pu = per_user[row['подход']]
    lo, hi = bootstrap_ci(pu, 'ndcg')
    row['NDCG test 95% ДИ'] = f'[{lo:.4f}, {hi:.4f}]'
    d, dlo, dhi = paired_ci(pu['ndcg'].to_numpy(), ref)
    row['Δ к ручным'] = f'{d:+.4f} [{dlo:+.4f}, {dhi:+.4f}]'
final_table = pl.DataFrame(final_rows)
final_table

Что видно в таблице:

- **Агрегаты по истории нужны.** Любой набор признаков лучше базовой модели на 0.005–0.010 NDCG@10 и лучше популярности на 0.013–0.018.
- **Автоматические наборы на уровне ручного.** Сетка с фильтром совпадает с ручным набором (+0.0001). Жадный поиск со случайным пулом лучше на 0.0012, но интервал парной разности задевает ноль, при этом признаков в пять раз меньше: 6 против 29.
- **LLM-пул проиграл.** Набор из жадного поиска с пулом от LLM значимо хуже ручного (−0.0024).
- **Найденные признаки дополняют ручные — на этой неделе.** Ручной набор плюс шесть признаков из LLM-поиска лучше ручного на 0.0020, и интервал [+0.0009, +0.0031] нуля не содержит. Однако на другой тестовой неделе этот прирост не воспроизводится (раздел 6.5).
- **Порядок на `val` и `test` не совпадает.** Сетка на `val` была хуже ручного набора на 0.004, на тесте они равны. Жадные поиски выбирали признаки по `val`, и у них падение от `val` к `test` больше: −0.0057 у LLM, −0.0045 у ручного набора, −0.0026 у случайного пула. Различия между подходами того же порядка, что и шум одной недели. Без парного бутстрепа их нельзя было бы интерпретировать.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
labels = [r['подход'] for r in final_rows if r['подход'] != 'ручные']
diffs = [paired_ci(per_user[n]['ndcg'].to_numpy(), ref) for n in labels]
ys = np.arange(len(labels))[::-1]
for y, (d, lo, hi) in zip(ys, diffs):
    color = COLORS[0] if lo > 0 else (COLORS[1] if hi < 0 else '#8a8984')
    ax.plot([lo, hi], [y, y], color=color, linewidth=2.5, solid_capstyle='round')
    ax.plot(d, y, 'o', color=color, markersize=8)
    ax.text(hi + 0.0005, y, f'{d:+.4f}', va='center', fontsize=9, color='#52514e')
ax.axvline(0, color='#52514e', linewidth=1)
ax.set_yticks(ys, labels)
ax.set_xlabel('Δ NDCG@10 относительно ручного набора')
ax.set_title('Тестовая неделя: разность с ручным набором, парный бутстреп, 95% ДИ', loc='left')
ax.grid(axis='y', visible=False)
fig.tight_layout()
plt.show()

Синий — значимо лучше ручного набора, оранжевый — значимо хуже, серый — разница в пределах шума.

### 6.1. Траектории поиска

NDCG@10 на `val` после каждого шага жадного поиска для двух генераторов пула. Горизонтальная линия — ручной набор на той же валидации. Траектории взяты из кэша поиска (`cache/search_*.json`), то есть из прогона, в котором поиск выполнялся, поэтому их начальная точка может немного отличаться от NDCG базовой модели в текущем прогоне.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
base_val = results[0][f'ndcg@{K}']
for hist, label, color in [(greedy_hist, 'случайный пул', COLORS[0]), (llm_hist, 'пул от LLM', COLORS[1])]:
    y = [base_val] + [h['ndcg'] for h in hist]
    ax.plot(range(len(y)), y, color=color, linewidth=2, marker='o', markersize=4, label=label)
ax.axhline(results[1][f'ndcg@{K}'], color='#8a8984', linestyle='--', linewidth=1.2, label='ручной набор')
ax.set_xlabel('шаг поиска')
ax.set_title(f'NDCG@{K} на val по шагам поиска', loc='left')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

### 6.2. Вычислительные затраты

Время первого прогона без кэша на ноутбуке автора (Intel i3-1215U, 4 потока для LightGBM, 2 для DuckDB), сохранено в `cache/timings.json`. Для поисков в него входит расчёт всех признаков-кандидатов и все обучения.

In [ ]:
def search_time(hist):
    return hist[-1]['elapsed'] if hist else float('nan')


n_cheap = lambda h: sum(len(x['tried']) for x in h)
cost = pl.DataFrame({
    'подход': ['ручные', 'сетка + фильтр', 'жадный, случайный пул', 'жадный, пул от LLM'],
    'признаков-кандидатов посчитано': [len(MANUAL), len(DFS_GRID),
                                       STEPS * SCREEN, len({t['spec'] for h in llm_hist for t in h['tried']})],
    'дешёвых обучений': [0, 0, n_cheap(greedy_hist), n_cheap(llm_hist)],
    'полных обучений': [1, len(dfs_rows), STEPS * TOP + 1, STEPS * TOP + 1],
    'время поиска, мин': [
        FIRST_RUN.get('ручные признаки, train + val', float('nan')) / 60,
        (FIRST_RUN.get('сетка: расчёт на train и одномерный отбор', 0) + FIRST_RUN.get('сетка: обучение моделей', 0)) / 60,
        search_time(greedy_hist) / 60,
        search_time(llm_hist) / 60,
    ],
})
cost

Ручной набор считается за секунды, но в эту цифру не входит время инженера, который его придумал. Сетка с фильтром стоит несколько минут, жадный поиск около 40 минут на ноутбуке без GPU. Больше половины времени LLM-поиска (24 из 45 минут) — ожидание ответов бесплатной модели. Платная модель или локальная модель на GPU сократила бы его до времени случайного поиска.

Масштабирование. Один признак — один SQL-запрос с группировкой: время растёт линейно с числом событий, на этом сэмпле 0.3–1 с. На полном датасете (в 10 раз больше событий) расчёт признаков подорожает примерно в 10 раз, а обучение LightGBM — пропорционально числу строк кандидатов. Поиск хорошо параллелится: кандидаты одного шага оцениваются независимо.

### 6.3. Что выбрали поиски

Сравниваем отобранные признаки по связям и по учёту времени. Ручной набор использует только окна 7 и 30 дней и всю историю.

In [ ]:
def describe(specs):
    return pl.DataFrame({
        'связь': [s.key.split(':')[0] if ':' not in s.key else s.key for s in specs],
        'время': [s.time for s in specs],
        'событие': [s.event for s in specs],
        'нормировка': [s.norm for s in specs],
    })


for name in ['жадный, случайный пул', 'жадный, пул от LLM']:
    d = describe(APPROACHES[name])
    print(f'\n{name}: {len(d)} признаков')
    print(d.group_by('время').len().sort('len', descending=True).rows())
    print(d.group_by('связь').len().sort('len', descending=True).rows())

Оба поиска выбрали одни и те же типы сигналов: качество тайтла (суммарное время просмотра, досматриваемость), популярность у пользователей того же пола, связи пользователя с жанрами. Признаков уровня пользователя не выбрал ни один. Различается учёт времени: случайный поиск взял окна от 7 до 60 дней, всю историю и затухание `h7`, а LLM-поиск — затухание `h14` в четырёх признаках из шести. Ни одно из найденных значений, кроме `w7` и `w30`, в ручном наборе не встречается.

### 6.4. LogLoss: бинарная постановка

В задании среди метрик есть LogLoss, а у `lambdarank` скоры не являются вероятностями. Для полноты обучаем те же наборы признаков с `objective='binary'`. Негативы в `train` прорежены, и без поправки вероятности были бы завышены. Поэтому каждый оставленный негатив получает вес «негативов у пользователя всего / оставлено». Число деревьев подбирается по early stopping на `val`, затем модель переобучается на полном `val` и предсказывает тест. Для масштаба приводим LogLoss константы, равной доле позитивов.

In [ ]:
from sklearn.metrics import log_loss

neg_total = candidates['train'].filter(pl.col('target') == 0).group_by('user_id').len('neg_total')
neg_kept = cand['train'].filter(pl.col('target') == 0).group_by('user_id').len('neg_kept')
w_train = (
    cand['train'].join(neg_total, on='user_id', how='left').join(neg_kept, on='user_id', how='left')
    .sort('row_id')
    .select(pl.when(pl.col('target') == 1).then(1.0).otherwise(pl.col('neg_total') / pl.col('neg_kept')))
    .to_series().to_numpy()
)
BIN_PARAMS = {k: v for k, v in LGB_PARAMS.items() if k not in ('eval_at', 'lambdarank_truncation_level')}
BIN_PARAMS.update(objective='binary', metric='binary_logloss')


def binary_eval(name, specs):
    names = BASE + [s.name for s in specs]
    X = matrices(specs, folds=('train', 'val'))
    dtr = lgb.Dataset(X['train'], cand['train']['target'].to_numpy(), weight=w_train, feature_name=names)
    dva = lgb.Dataset(X['val'], cand['val']['target'].to_numpy(), reference=dtr)
    m = lgb.train(BIN_PARAMS, dtr, 2000, valid_sets=[dva], callbacks=[lgb.early_stopping(50, verbose=False)])
    n_trees = m.best_iteration
    del X, dtr, dva
    gc.collect()
    X = matrices(specs, folds=('val', 'test'))
    m = lgb.train(BIN_PARAMS, lgb.Dataset(X['val'], cand['val']['target'].to_numpy(), feature_name=names), n_trees)
    p = m.predict(X['test'])
    del X
    gc.collect()
    y = cand['test']['target'].to_numpy()
    res = evaluate(p, 'test')
    return {'подход': name, 'LogLoss test': log_loss(y, p), 'AUC test': res['auc'], f'NDCG@{K} test': res[f'ndcg@{K}']}


y_test = cand['test']['target'].to_numpy()
with timer('бинарные модели'):
    bin_rows = [{'подход': 'константа (доля позитивов)',
                 'LogLoss test': log_loss(y_test, np.full(len(y_test), y_test.mean())),
                 'AUC test': 0.5, f'NDCG@{K} test': None}]
    for name in ['только базовые', 'ручные', 'жадный, случайный пул', 'ручные + жадный (LLM)']:
        bin_rows.append(binary_eval(name, APPROACHES[name]))
pl.DataFrame(bin_rows)

Все модели заметно лучше константы: LogLoss снижается с 0.0417 до 0.036. Порядок подходов тот же, что по NDCG: ручные + найденные признаки (0.0357), жадный поиск (0.0360), ручные (0.0364), базовые (0.0370). Различия малы, потому что позитивов всего 0.7% и основная часть LogLoss приходится на пары, которые любая модель уверенно считает нерелевантными.

Бинарная модель ранжирует немного хуже `lambdarank`: NDCG@10 у ручного набора 0.1504 против 0.1524. Это ожидаемо, `lambdarank` напрямую оптимизирует порядок внутри пользователя.

### 6.5. Устойчивость: другая тестовая неделя

Разности между подходами порядка 0.001–0.002 NDCG, а тестовая неделя одна. Проверяем, повторится ли картина на другой неделе. Берём разбиение, сдвинутое на три недели назад: `train` 12–18.07, `val` 19–25.07, `test` 26.07–01.08. Ни одна из этих недель не участвовала в поиске признаков, который смотрел на недели 02.08–15.08. Наборы признаков заморожены, протокол обучения тот же, что в разделе 6.

In [ ]:
for f in ['valtr']:
    del XB[f], cand[f]
    con.execute(f'DROP TABLE ev_{f}; DROP TABLE cand_{f}')
gc.collect()

FOLDS_B = {
    'train_b': TEST_START - timedelta(days=35),
    'val_b': TEST_START - timedelta(days=28),
    'test_b': TEST_START - timedelta(days=21),
}
with timer('вторая неделя: кандидаты и таблицы'):
    for fold, start in FOLDS_B.items():
        candidates[fold] = cached(f'candidates_{fold}_{N_USERS}_{N_POP}_{N_KNN}',
                                  lambda s=start: build_candidates(log, s))
        setup_fold(fold, start)
        cand[fold] = make_cand_table(fold, sample_train(candidates[fold]) if fold == 'train_b' else candidates[fold])
        XB[fold] = base_matrix(fold)
        WARM[fold] = split(log, start)[1]
    setup_fold('valtr_b', FOLDS_B['val_b'])
    cand['valtr_b'] = make_cand_table('valtr_b', sample_train(candidates['val_b']))
    XB['valtr_b'] = base_matrix('valtr_b')

with timer('вторая неделя: оценка'):
    per_user_b = {'популярность': evaluate(XB['test_b'][:, BASE.index('pop_score')], 'test_b', per_user=True)[1].sort('user_id')}
    for name, specs in APPROACHES.items():
        _, per_user_b[name] = final_eval(name, specs, folds=('train_b', 'val_b', 'valtr_b', 'test_b'))

ref_b = per_user_b['ручные']['ndcg'].to_numpy()
robust = []
for name in per_user_b:
    d, lo, hi = paired_ci(per_user_b[name]['ndcg'].to_numpy(), ref_b)
    d0, lo0, hi0 = paired_ci(per_user[name]['ndcg'].to_numpy(), ref)
    robust.append({'подход': name, f'NDCG@{K} 26.07': per_user_b[name]['ndcg'].mean(),
                   'Δ к ручным, 26.07': f'{d:+.4f} [{lo:+.4f}, {hi:+.4f}]',
                   'Δ к ручным, 16.08': f'{d0:+.4f} [{lo0:+.4f}, {hi0:+.4f}]'})
print(f'пользователей в тесте 26.07: {len(ref_b):,}')
pl.DataFrame(robust)

На неделе 26.07 картина частично другая, и это важная поправка к выводам по одной неделе.

- **Устойчиво.** Агрегаты по истории нужны: без них NDCG ниже на 0.008–0.009 на обеих неделях. Набор из LLM-поиска хуже ручного на обеих неделях (−0.0024 и −0.0031). Сетка с фильтром не лучше ручного набора, а на 26.07 значимо хуже (−0.0044), что согласуется с её отставанием на `val`.
- **Неустойчиво.** Жадный поиск со случайным пулом: +0.0012 на 16.08 и −0.0016 на 26.07, оба интервала задевают ноль. Корректный вывод — паритет с ручным набором, а не превосходство.
- **Не воспроизвелось.** Прирост от объединения ручных и найденных признаков (+0.0020 на 16.08) на 26.07 исчезает: −0.0004, интервал содержит ноль. Признаки с затуханием `h14` отбирались по неделям 02–15.08 и, вероятно, отражают особенности именно этого периода.

Разница в 0.002 NDCG между наборами признаков сопоставима с изменчивостью между неделями. Одной тестовой недели недостаточно, чтобы утверждать превосходство одного набора над другим.

### 6.6. Разброс по seed LightGBM

Доверительные интервалы выше учитывают только случайность выборки пользователей. Но у обучения LightGBM есть своя случайность: подвыборки строк (`bagging`) и признаков (`feature_fraction`) зависят от `seed`, а многопоточное обучение на разных процессорах даёт немного разные деревья. Поэтому повторяем финальную оценку на обеих тестовых неделях с пятью значениями `seed` (42 — основной прогон) и смотрим, у каких подходов разность с ручным набором сохраняет знак.

In [ ]:
# Недели считаем по очереди: таблицы обеих недель одновременно не помещаются в память ноутбука.
for f in ['train', 'test', 'train_b', 'val_b', 'test_b']:
    candidates.pop(f, None)
gc.collect()


def drop_folds(folds):
    for f in folds:
        del XB[f], cand[f]
        con.execute(f'DROP TABLE ev_{f}; DROP TABLE cand_{f}')
    gc.collect()


def prepare_week(week):
    if week == '16.08':
        drop_folds(['train_b', 'val_b', 'valtr_b', 'test_b'])
        setup_fold('valtr', FOLDS['val'])
        cand['valtr'] = make_cand_table('valtr', sample_train(candidates['val']))
        XB['valtr'] = base_matrix('valtr')


SEEDS = [42, 1, 2, 3, 4]
WEEKS = {'26.07': ('train_b', 'val_b', 'valtr_b', 'test_b'), '16.08': ('train', 'val', 'valtr', 'test')}
seed_rows = []
with timer('разброс по seed'):
    for week, folds in WEEKS.items():
        prepare_week(week)
        for seed in SEEDS:
            LGB_PARAMS['seed'] = seed
            ndcg = {name: final_eval(name, specs, folds=folds)[1]['ndcg'].mean() for name, specs in APPROACHES.items()}
            for name, v in ndcg.items():
                seed_rows.append({'неделя': week, 'seed': seed, 'подход': name, 'ndcg': v, 'delta': v - ndcg['ручные']})
LGB_PARAMS['seed'] = SEED
seeds_df = pl.DataFrame(seed_rows)

order = list(APPROACHES)
summary = (
    seeds_df.group_by('неделя', 'подход')
    .agg(
        pl.col('ndcg').mean().alias('NDCG среднее'),
        pl.col('ndcg').std().alias('NDCG σ'),
        pl.col('delta').mean().alias('Δ к ручным, среднее'),
        pl.col('delta').min().alias('Δ min'),
        pl.col('delta').max().alias('Δ max'),
        (pl.col('delta') > 0).sum().alias('seed с Δ > 0'),
    )
    .with_columns(pl.col('подход').replace_strict(order, list(range(len(order)))).alias('_o'))
    .sort('неделя', '_o', descending=[True, False])
    .drop('_o')
)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
names_s = [n for n in order if n != 'ручные']
for k, (week, color) in enumerate([('16.08', COLORS[0]), ('26.07', COLORS[1])]):
    for i, name in enumerate(names_s):
        d = seeds_df.filter((pl.col('неделя') == week) & (pl.col('подход') == name))['delta'].to_numpy()
        y = len(names_s) - 1 - i + (0.15 if k == 0 else -0.15)
        ax.scatter(d, np.full(len(d), y), color=color, s=28, zorder=3, label=f'неделя {week}' if i == 0 else None)
ax.axvline(0, color='#52514e', linewidth=1)
ax.set_yticks(range(len(names_s)), names_s[::-1])
ax.set_xlabel('Δ NDCG@10 относительно ручного набора')
ax.set_title('Разность с ручным набором при пяти seed LightGBM (точка — один seed)', loc='left')
ax.legend(frameon=False, loc='lower left')
ax.grid(axis='y', visible=False)
fig.tight_layout()
plt.show()

SEED_CONCLUSION

## 7. Выводы

**Можно ли автоматизировать построение признаков без потери качества?** На KION — да, при условии что человек задаёт пространство признаков. Жадный поиск по грамматике из 7 389 спецификаций нашёл набор из 6 признаков, который на двух тестовых неделях не отличается от ручного набора из 29 значимо (+0.0012 и −0.0016 NDCG@10, оба интервала содержат ноль). Превзойти ручной набор устойчиво не удалось: прирост +0.0020 от объединения ручных и найденных признаков на первой тестовой неделе не воспроизвёлся на второй.

**Что сработало**

- Грамматика признаков, которая компилируется в SQL. Ручные признаки оказываются частным случаем пространства, сравнение подходов становится честным, а утечки исключены конструктивно: единственный источник событий — таблица с отсечкой по дате, это проверено тестом с удалением будущего.
- Жадный отбор с учётом текущего набора. Он находит признаки, которые добавляют новую информацию, а не дублируют уже имеющиеся.
- Затухание и нестандартные окна. Поиски выбирали `h7`, `h14`, `h60`, `w60` — значения, которые человек обычно не перебирает.

**Что не сработало**

- Одномерный фильтр поверх сетки (DFS). Он набирает близкие варианты популярности, которые по отдельности сильные, а вместе почти бесполезны.
- LLM как генератор пула в текущем виде. Бесплатная Nemotron 3 Ultra соблюдает формат (89% допустимых спецификаций), но предлагает узкий набор «разумных» вариантов, игнорирует подсказку про бесполезность признаков пользователя и иногда возвращает неполный список. Знание предметной области не компенсировало потерю разнообразия.

**Ограничения**

- Две тестовые недели и сэмпл 100 тыс. пользователей. Разности между подходами порядка 0.001–0.002 NDCG, это сопоставимо с изменчивостью между неделями: на второй неделе знак разности у жадного поиска сменился. Выводы о «не хуже» надёжнее выводов о «лучше».
- Отбор по `val` переобучается: у наборов, выбранных поиском, падение от `val` к `test` больше, чем у остальных. Валидация на нескольких неделях уменьшила бы этот эффект.
- Формат данных: одна строка на пару и нет показов. Прямые признаки «пользователь × объект» и CTR недоступны, поэтому вывод относится к косвенным связям.
- Потолок кандидатов 44% позитивов: все ранкеры работают на одном и том же наборе и улучшают только порядок внутри него.
- Гиперпараметры LightGBM подобраны один раз на ручном наборе. LLM-поиск выполнен одной моделью и одним прогоном, а ответы модели стохастичны.
- SEED_LIMIT

**Что можно сделать дальше**

- Пропускать предложения LLM через тот же одномерный отбор, что и случайный пул: просить 120 кандидатов и оставлять 24 лучших. Это убрало бы признаки пользователя и выровняло бюджет.
- Сделать полупериод затухания непрерывным параметром и искать его байесовской оптимизацией, а не по сетке.
- Повторить сам поиск признаков с разными seed: сейчас каждый поиск выполнен один раз, и неизвестно, насколько стабилен выбранный набор.
- Отбирать признаки по нескольким неделям валидации. Раздел 6.5 показывает, что признаки, отобранные по двум соседним неделям, частично подстраиваются под их особенности.
- Добавить скор последовательной модели (SASRec) как признак и проверить, дополняет ли выученное представление истории найденные агрегаты.